In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
!nvidia-smi

Mon Sep 28 15:44:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1050        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P8            N/A  / 5001W |       7MiB /   3072MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Характеристики компьютера
Сначала CPU, оперативная память, накопители и программная среда.
В таблице дисков `ROTA=0` означает SSD, `ROTA=1` — HDD.

In [3]:
import platform

print("=== Программное обеспечение ===")
print("ОС:", platform.freedesktop_os_release()["PRETTY_NAME"])
print("Python:", platform.python_version())

print("\n=== Процессор ===")
!LC_ALL=C lscpu | grep -E "Model name|^CPU\(s\):|Core\(s\) per socket|Thread\(s\) per core"

print("\n=== Оперативная память ===")
print("Столбцы: total, used, free, shared, buff/cache, available")
!LC_ALL=C free -h | grep "^Mem:"

print("\n=== Накопители ===")
print("ROTA: 0 — SSD/NVMe, 1 — HDD")
!lsblk -d -e 7 -o NAME,MODEL,SIZE,ROTA

print("\n=== Компилятор C++ ===")
!g++ --version | head -1

print("\n=== CUDA ===")
!nvcc --version | grep release

=== Программное обеспечение ===
ОС: Ubuntu 22.04.5 LTS
Python: 3.10.12

=== Процессор ===


CPU(s):                                  12
Model name:                              Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
Thread(s) per core:                      2
Core(s) per socket:                      6



=== Оперативная память ===
Столбцы: total, used, free, shared, buff/cache, available
Mem:            15Gi       8.3Gi       1.6Gi       1.4Gi       5.5Gi       6.3Gi



=== Накопители ===
ROTA: 0 — SSD/NVMe, 1 — HDD
NAME    MODEL                SIZE ROTA
sda     WDC WD10SPZX-24Z10 931.5G    1
nvme0n1 HFM256GDHTNG-8310A 238.5G    0



=== Компилятор C++ ===
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0



=== CUDA ===
Cuda compilation tools, release 12.8, V12.8.93


## Программа для GPU
Ядро `vector_min` совпадает с ядром решения: каждый поток проходит свой участок массива с шагом сетки.
Для каждого размера и каждой конфигурации сетки ядро выполняется **один раз**.
Перед каждой конфигурацией `fill` подготавливает одинаковые исходные данные на GPU.
`correct` проверяет все элементы после вычисления; аргумент `info` выводит характеристики GPU.

Измеряем только вычисления: заполнение, выделение и миграция памяти, проверка и печать не входят во время.

In [4]:
gpu_code = r"""
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>

void check(cudaError_t error) {
    if (error != cudaSuccess) {
        fprintf(stderr, "%s\n", cudaGetErrorString(error));
        exit(1);
    }
}

__global__ void vector_min(double* a, const double* b, int n) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int step = blockDim.x * gridDim.x;
    for (int i = index; i < n; i += step) {
        if (a[i] > b[i]) {
            a[i] = b[i];
        }
    }
}

__global__ void fill(double* a, double* b, int n) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int step = blockDim.x * gridDim.x;
    for (int i = index; i < n; i += step) {
        a[i] = i % 1024 - 512 + 0.25;
        b[i] = (i * 17) % 1024 - 500 + 0.5;
    }
}

bool correct(double* a, double* b, int n) {
    for (int i = 0; i < n; i++) {
        double x = i % 1024 - 512 + 0.25;
        double y = (i * 17) % 1024 - 500 + 0.5;
        if (a[i] != (x < y ? x : y) || b[i] != y) return false;
    }
    return true;
}

void print_properties() {
    cudaDeviceProp p;
    check(cudaGetDeviceProperties(&p, 0));
    printf("GPU: %s\n", p.name);
    printf("Compute capability: %d.%d\n", p.major, p.minor);
    printf("Глобальная память: %.2f MiB\n", p.totalGlobalMem / 1048576.0);
    printf("Разделяемая память на блок: %zu байт\n", p.sharedMemPerBlock);
    printf("Константная память: %zu байт\n", p.totalConstMem);
    printf("32-битные регистры на блок: %d\n", p.regsPerBlock);
    printf("Максимум нитей на блок: %d\n", p.maxThreadsPerBlock);
    printf("Максимальные размеры блока: %d x %d x %d\n",
           p.maxThreadsDim[0], p.maxThreadsDim[1], p.maxThreadsDim[2]);
    printf("Максимальные размеры сетки: %d x %d x %d\n",
           p.maxGridSize[0], p.maxGridSize[1], p.maxGridSize[2]);
    printf("Мультипроцессоры (SM): %d\n", p.multiProcessorCount);
    printf("Максимум нитей на SM: %d\n", p.maxThreadsPerMultiProcessor);
    printf("Размер warp: %d\n", p.warpSize);
    printf("Частота GPU из CUDA properties: %.0f МГц\n", p.clockRate / 1000.0);
    printf("Шина памяти: %d бит\n", p.memoryBusWidth);
}

int main(int argc, char** argv) {
    if (argc > 1) {
        print_properties();
        return 0;
    }

    int sizes[] = {1024, 1048576, 33554431};
    int configs[][2] = {{1, 32}, {16, 128}, {64, 128}, {128, 256},
                       {256, 256}, {512, 256}, {512, 512}, {1024, 1024}};
    cudaEvent_t start, stop;
    check(cudaEventCreate(&start));
    check(cudaEventCreate(&stop));
    printf("n,blocks,threads,ms\n");

    for (int n : sizes) {
        double *a, *b;
        check(cudaMallocManaged(&a, n * sizeof(double)));
        check(cudaMallocManaged(&b, n * sizeof(double)));
        for (auto& config : configs) {
            int blocks = config[0];
            int threads = config[1];

            fill<<<256, 256>>>(a, b, n);
            check(cudaDeviceSynchronize());

            check(cudaEventRecord(start));
            vector_min<<<blocks, threads>>>(a, b, n);
            check(cudaGetLastError());
            check(cudaEventRecord(stop));
            check(cudaEventSynchronize(stop));

            float ms;
            check(cudaEventElapsedTime(&ms, start, stop));
            if (!correct(a, b, n)) return 1;
            printf("%d,%d,%d,%.9f\n", n, blocks, threads, ms);
        }
        check(cudaFree(a));
        check(cudaFree(b));
    }
    check(cudaEventDestroy(start));
    check(cudaEventDestroy(stop));
}
"""

## Последовательная программа для CPU
Те же массивы и операция минимума, обычный цикл C++ без CUDA и OpenMP.
Для каждого размера массивы заполняются, а функция `vector_min` выполняется **один раз**.
Время измеряется через `std::chrono`. Проверка всех элементов выполняется после вычисления и не входит в замер.

In [5]:
cpu_code = r"""
#include <chrono>
#include <cstdio>

void vector_min(double* a, const double* b, int n) {
    for (int i = 0; i < n; i++) {
        if (a[i] > b[i]) a[i] = b[i];
    }
}

int main() {
    int sizes[] = {1024, 1048576, 33554431};
    printf("n,ms\n");
    for (int n : sizes) {
        double* a = new double[n];
        double* b = new double[n];
        for (int i = 0; i < n; i++) {
            a[i] = i % 1024 - 512 + 0.25;
            b[i] = (i * 17) % 1024 - 500 + 0.5;
        }

        auto start = std::chrono::steady_clock::now();
        vector_min(a, b, n);
        auto stop = std::chrono::steady_clock::now();
        double ms = std::chrono::duration<double, std::milli>(stop - start).count();

        for (int i = 0; i < n; i++) {
            double x = i % 1024 - 512 + 0.25;
            double y = (i * 17) % 1024 - 500 + 0.5;
            if (a[i] != (x < y ? x : y) || b[i] != y) return 1;
        }
        printf("%d,%.9f\n", n, ms);
        delete[] a;
        delete[] b;
    }
}
"""

## Сборка, характеристики GPU и все эксперименты
На GPU перебираем восемь конфигураций от `<<<1,32>>>` до `<<<1024,1024>>>`.
Размеры: 1024, 1 048 576 и предельный 2²⁵ − 1 = 33 554 431.
Каждый случай запускается один раз: всего **24 измерения GPU и 3 измерения CPU**.

In [6]:
import csv
import subprocess
import tempfile

with tempfile.TemporaryDirectory() as folder:
    with open(folder + "/gpu.cu", "w") as file:
        file.write(gpu_code)
    with open(folder + "/cpu.cpp", "w") as file:
        file.write(cpu_code)

    subprocess.run(["/usr/local/cuda-12.8/bin/nvcc", "-O2", "-std=c++11",
                    "-arch=native", "gpu.cu", "-o", "gpu"], cwd=folder, check=True)
    subprocess.run(["g++", "-O2", "-std=c++11", "cpu.cpp", "-o", "cpu"],
                   cwd=folder, check=True)

    print(subprocess.check_output(["./gpu", "info"], cwd=folder, text=True), flush=True)
    print("Запуск GPU...", flush=True)
    gpu_output = subprocess.check_output(["./gpu"], cwd=folder, text=True)
    print("Запуск CPU...", flush=True)
    cpu_output = subprocess.check_output(["./cpu"], cwd=folder, text=True)

gpu = list(csv.DictReader(gpu_output.splitlines()))
cpu = list(csv.DictReader(cpu_output.splitlines()))
print("Все проверки результата пройдены.")

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


GPU: NVIDIA GeForce GTX 1050
Compute capability: 6.1
Глобальная память: 3011.81 MiB
Разделяемая память на блок: 49152 байт
Константная память: 65536 байт
32-битные регистры на блок: 65536
Максимум нитей на блок: 1024
Максимальные размеры блока: 1024 x 1024 x 64
Максимальные размеры сетки: 2147483647 x 65535 x 65535
Мультипроцессоры (SM): 6
Максимум нитей на SM: 2048
Размер warp: 32
Частота GPU из CUDA properties: 1442 МГц
Шина памяти: 96 бит



Запуск GPU...


Запуск CPU...


Все проверки результата пройдены.


## Время GPU
Для каждого размера приведено время одного запуска каждой конфигурации сетки.
Все времена указаны в миллисекундах. Округление применяется только при выводе таблиц.

In [7]:
import pandas as pd

sizes = [1024, 1048576, 33554431]
configs = [
    (1, 32),
    (16, 128),
    (64, 128),
    (128, 256),
    (256, 256),
    (512, 256),
    (512, 512),
    (1024, 1024)
]

gpu_times = {}
for row in gpu:
    key = (int(row["n"]), int(row["blocks"]), int(row["threads"]))
    gpu_times[key] = float(row["ms"])

for n in sizes:
    rows = []
    for blocks, threads in configs:
        rows.append({
            "Конфигурация <<<блоки, нити>>>": f"<<<{blocks}, {threads}>>>",
            "Время GPU, мс": gpu_times[n, blocks, threads]
        })

    print(f"\nGPU: размер входных данных n = {n}")
    display(pd.DataFrame(rows).round(6))


GPU: размер входных данных n = 1024


,"Конфигурация <<<блоки, нити>>>","Время GPU, мс"
0,"<<<1, 32>>>",0.094208
1,"<<<16, 128>>>",0.012288
2,"<<<64, 128>>>",0.011264
3,"<<<128, 256>>>",0.011264
4,"<<<256, 256>>>",0.014336
5,"<<<512, 256>>>",0.009216
6,"<<<512, 512>>>",0.016384
7,"<<<1024, 1024>>>",0.030720



GPU: размер входных данных n = 1048576


,"Конфигурация <<<блоки, нити>>>","Время GPU, мс"
0,"<<<1, 32>>>",10.957824
1,"<<<16, 128>>>",0.331776
2,"<<<64, 128>>>",0.295936
3,"<<<128, 256>>>",0.293888
4,"<<<256, 256>>>",0.288768
5,"<<<512, 256>>>",0.288768
6,"<<<512, 512>>>",0.316416
7,"<<<1024, 1024>>>",0.314368



GPU: размер входных данных n = 33554431


,"Конфигурация <<<блоки, нити>>>","Время GPU, мс"
0,"<<<1, 32>>>",322.386932
1,"<<<16, 128>>>",9.765888
2,"<<<64, 128>>>",9.177088
3,"<<<128, 256>>>",9.111552
4,"<<<256, 256>>>",9.147392
5,"<<<512, 256>>>",9.094144
6,"<<<512, 512>>>",9.092096
7,"<<<1024, 1024>>>",9.069568


## Время CPU
Каждая строка — время одного запуска последовательной программы для указанного размера массива.
Таблица показывает, как меняется время при увеличении `n`.

In [8]:
cpu_times = {}
rows = []

for row in cpu:
    n = int(row["n"])
    ms = float(row["ms"])
    cpu_times[n] = ms
    rows.append({"Размер n": n, "Время CPU, мс": ms})

cpu_table = pd.DataFrame(rows)
display(cpu_table.round(6))

,Размер n,"Время CPU, мс"
0,1024,0.001966
1,1048576,5.929520
2,33554431,65.881123


## Сравнение CPU и GPU
Для каждого `n` выбираем конфигурацию GPU с наименьшим временем среди восьми проверенных в этом запуске.
Сравниваем её время со временем CPU для того же размера.


In [9]:
best_gpu = {}
comparison_rows = []

for n in sizes:
    best_blocks, best_threads = configs[0]
    for blocks, threads in configs:
        if gpu_times[n, blocks, threads] < gpu_times[n, best_blocks, best_threads]:
            best_blocks, best_threads = blocks, threads

    best_gpu[n] = (best_blocks, best_threads)
    cpu_ms = cpu_times[n]
    gpu_ms = gpu_times[n, best_blocks, best_threads]

    comparison_rows.append({
        "Размер n": n,
        "Лучшая конфигурация GPU": f"<<<{best_blocks}, {best_threads}>>>",
        "Время CPU, мс": cpu_ms,
        "Время GPU, мс": gpu_ms,
        "CPU / GPU": cpu_ms / gpu_ms
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(6))

,Размер n,Лучшая конфигурация GPU,"Время CPU, мс","Время GPU, мс",CPU / GPU
0,1024,"<<<512, 256>>>",0.001966,0.009216,0.213325
1,1048576,"<<<256, 256>>>",5.929520,0.288768,20.533855
2,33554431,"<<<1024, 1024>>>",65.881123,9.069568,7.263976
